# Level 1 · Leaderboard your agent

You have a method — Meta-Harness, DeLM, GEPA, a memory scheme, your own scaffold. This notebook
turns it into **one row of the leaderboard**, comparable to every published row, in about as
many lines as it takes to describe it.

The service hosts the *environments* and the *graders*. You never stand up a Docker gym, boot a
sandbox, or download a dataset — and you are **not** running Terminal-Bench or any upstream
suite. You are running **EvoHarnessBench-EOG** and **EvoHarnessBench-ALE**, on the same task
sets and the same verifiers the paper's tables were produced from.

### The row you are producing

```
                    EvoHarnessBench-EOG (454)          EvoHarnessBench-ALE (63)       Overall
                Pass %   Score   ⏱ h   Tok M      Pass %   Score   ⏱ h   Tok M      Pass Rate
ReAct / Codex    30.2     60.0   28.6   75.8       12.2     30.0    9.2  100.8         28.1
Meta-Harness     35.2     65.8   27.3   99.0       11.1     30.4   11.2  107.8         32.3
DeLM             20.3     51.4   97.7  179.1        6.2     23.7   16.7   30.9         18.7
GEPA             31.9     65.9   31.9  100.2       11.1     30.9    9.6   86.9         29.3
your method       ...      ...    ...    ...        ...      ...    ...    ...          ...
```

Two accuracy columns, because they answer different questions and methods trade between them:

| column | definition | reads as |
|---|---|---|
| **Pass %** | strict — *every* verifier on the task passed | did it finish the job |
| **Score** | mean verifier pass rate | how much of the job it got right |
| **⏱ h** | summed **agent** duration, not wall clock | what it cost in time |
| **Tok M** | input + output tokens, millions | what it cost in tokens |
| **Overall** | Pass, micro-averaged over EOG + ALE together | the single ranking number |

The gap between Pass and Score is partial credit. The tools leader passes 38.6% of tasks
outright while averaging 68.9% of the checks — the difference between finishing a workflow and
getting most of it right. Reporting only one of them hides which kind of method you built.

**Ten minutes to a smoke test; the publishable row costs real money and hours** — §6 sizes it
before you start.

### Before you start: get a key

This service is gated. Sign in at [the MAS-Orchestra demo](https://mas-orchestra.salesforceresearch.ai/mas_r1/demo/),
then open **MyAuthtoken** from the signed-in view — that page issues your key. It is *yours*:
keep it out of shared notebooks and out of version control. The Setup cell prompts for it, so
you never have to paste it into a cell that might get committed.

Two different keys are in play, and confusing them is the most common setup failure:

| variable | what it opens | where it comes from |
|---|---|---|
| `EVAL_SERVICE_API_KEY` | this evaluation service | MyAuthtoken, at the link above |
| `OPENAI_API_KEY` | whatever **your method** thinks with | your own OpenAI account |

Your OpenAI key is only ever seen by the agent you hand it to. The one deliberate exception is
the **provided harnesses** (`"react"`, `"codex"`, Level 3): they run *on the service*, so the
key travels with that request. They **require** it — the service hosts the environment and the
grader, but does not pay for your inference and will not fall back to its own key.

### The three tutorials

| | | |
|---|---|---|
| **1 · Leaderboard your agent** | your method → one comparable row | [Colab](https://colab.research.google.com/drive/1xJEpRf_s0zG-M9QynS3MBk7Nkr-xB11r) ← **you are here** |
| **2 · Continual learning** | the full performance matrix, BWT, FWT, adaptation | [Colab](https://colab.research.google.com/drive/1vrcGelN9GmwiCK25c6qZNG3Z0sHWxE5o) |
| **3 · Harnesses & modes** | the provided agents, every mode, standard non-evolving eval, full reference | [Colab](https://colab.research.google.com/drive/1mYQEDCVFStXMRWYI2hpEGBXwyRx1NSFj) |

In [ ]:
# Setup, in one cell. The client SDK is served BY the service (GET /sdk), so there is no
# PyPI account, no repo checkout, and no service URL anywhere in your code afterwards.
import getpass, importlib, json, os, re, subprocess, sys, tempfile, urllib.error, urllib.request
from functools import partial

SERVICE_URL = os.environ.get("EVAL_SERVICE_URL",
                             "https://educator-marrow-cultural.ngrok-free.dev")

if not os.environ.get("EVAL_SERVICE_API_KEY"):
    os.environ["EVAL_SERVICE_API_KEY"] = getpass.getpass("Eval service key (MyAuthtoken): ")
SDK_HEADERS = {"Authorization": f"Bearer {os.environ['EVAL_SERVICE_API_KEY']}",
               "ngrok-skip-browser-warning": "true"}
def sdk_read(request):
    try:
        with urllib.request.urlopen(request) as response:
            return response.read()
    except urllib.error.HTTPError as exc:
        try: detail = json.loads(exc.read()).get("detail", str(exc))
        except Exception: detail = str(exc)
        raise SystemExit(f"SDK download denied: {detail}") from None


def _sdk_current(minimum=(0, 16, 0)) -> bool:
    # A stale pre-installed copy would shadow the service's wheel, so check the API and
    # version rather than mere importability.
    try:
        import simple_agentic_evals as m
        v = tuple(int(x) for x in str(getattr(m, "__version__", "0")).split(".")[:3])
        return hasattr(m, "run_leaderboard") and v >= minimum
    except Exception:
        return False


subprocess.run([sys.executable, "-m", "pip", "install", "-q", "openai"], check=True)
if not _sdk_current():
    req = urllib.request.Request(f"{SERVICE_URL}/sdk", headers=SDK_HEADERS)
    manifest = json.loads(sdk_read(req))
    wheel = os.path.join(tempfile.mkdtemp(prefix="eval_sdk_"), manifest["filename"])
    wheel_req = urllib.request.Request(f"{SERVICE_URL}{manifest['path']}", headers=SDK_HEADERS)
    with open(wheel, "wb") as output:
        output.write(sdk_read(wheel_req))
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade",
                    "--force-reinstall", "--no-deps", wheel], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "httpx"], check=True)
    for k in [k for k in list(sys.modules) if k.startswith("simple_agentic_evals")]:
        del sys.modules[k]
    importlib.invalidate_caches()

if not os.environ.get("OPENAI_API_KEY"):        # the key YOUR agent thinks with
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API key: ")

from openai import OpenAI
from simple_agentic_evals import (EvalClient, ServiceError, run_benchmark, run_leaderboard,
                                  CommandAgent, react_agent, acp_codex_agent, to_openai_tools)

client    = EvalClient()     # the endpoint is baked into the wheel -- no URL to paste
oai       = OpenAI()         # only for the bring-your-own-agent examples
LLM_MODEL = os.environ.get("EVAL_LLM_MODEL", "gpt-4o-mini")
# Every /v1/* route needs the key, health included -- a rejected key means you
# cannot use this service, so it should not report itself healthy. So this one
# call both connects and proves the key, and a bad key fails HERE with the
# message that says where to get a new one.
try:
    _h = client.health()
except ServiceError as e:
    raise SystemExit(f"cannot use the eval service: {e.detail}") from None
print("connected ->", _h)


def harness(fn, *args, **kwargs):
    """Call a provided harness, or say why this deployment can't and return None.

    react_agent / acp_codex_agent execute ON THE SERVICE, so they depend on what that host
    has installed. Bringing your own agent needs none of it.
    """
    try:
        return fn(*args, **kwargs)
    except ServiceError as e:
        # The service prefixes a banner and appends the subprocess's stderr tail, so the
        # LAST line is the actual cause -- the first is just "...stderr tail:".
        lines = [ln for ln in (e.detail or "").strip().splitlines() if ln.strip()]
        print(f"  [harness unavailable] {e.status_code}: {(lines[-1] if lines else e)!s:.170}")
        if len(lines) > 1:
            print(f"     ({len(lines)} more lines of server traceback in e.detail)")
        return None

## 0. Health check — run this first

The service is a **deployment**, not a library. The environment, the graders and the two
provided harnesses all execute on *that host*, so what actually works depends on what it has
installed. The failure mode is quiet: an unavailable harness returns **without acting**, and
its run then scores the do-nothing baseline — which reads as a real (bad) result rather than
an error. Verify the deployment before you trust a number from it.

Each row is one capability, and a red row tells you what you lose, not what you did wrong:

| row | what it proves | if it's red |
|---|---|---|
| `reachable`, `sdk` | service is up; your client matches its served wheel | nothing works / re-run Setup |
| `catalog`, `eog tasks` | both benchmarks advertised, the slice is populated *and its count matches the catalog* | that data isn't deployed |
| `ale tasks` | the ALE **denominator**: exactly the tasks a published number counts | you can't compare an ALE score to anything |
| `eog env+grade` | session provisioning, the gym MCP proxy, SQL verifiers | all of EOG (§2) |
| `resources` | the evolving axis — `oracle` ⊆ `accumulative` | the evolving modes (§6) |
| `react`, `codex` | the **server-side** harnesses, and the host packages they need | those harnesses only — your own loop (§2) needs neither |
| `ale` | input staging, artifact submit, the task's own `evaluate()` | all of ALE (§5) |

The harness rows are the only way to spot a server missing `langchain` or the Codex CLI. A
harness that *runs* but stops with `error` is counted **red** — that's the silent failure this
section exists to catch. Set `DEEP = False` to skip the three rows that cost LLM calls or boot
a sandbox.

The task-count rows are about a subtler kind of wrong. ALE ships 152 tasks, but 47 of them need
a non-Linux VM and a few more die in their own loader on any given host, so only a subset can
produce a number at all — and a task that provisions and returns `0.0` for an environmental
reason is indistinguishable from an agent that tried and failed. The service therefore serves
only the tasks its exclusion manifest says are measurable, and `ale tasks` re-derives that
count from the catalog and fails if the two disagree. A green row means an ALE score you can
put next to someone else's.

In [ ]:
import urllib.error
DEEP = True    # False -> skip the rows that cost LLM calls or boot an ALE sandbox

# Fixed reference slices, so this stays a self-test of the DEPLOYMENT whatever you set in 1.
EOG_PROBE = ("evovling_tools", "eog", 1, "test", "hr")   # dataset, benchmark, version, split, domain
ALE_PROBE = ("evovling_tools", "ale", 1, "legal/agora_governance_classify_instance_1")
_rows = []


def _check(name, fn, lose="", deep=False):
    if deep and not DEEP:
        _rows.append((name, "SKIP"))
        print(f"  [SKIP] {name:14} DEEP=False")
        return
    try:
        detail, status = fn(), "OK"
    except ServiceError as e:                    # the service said no -- report why
        lines = [ln for ln in (e.detail or "").strip().splitlines() if ln.strip()]
        detail, status = f"{e.status_code}: {(lines[-1] if lines else e)!s:.100}", "FAIL"
    except Exception as e:
        detail, status = f"{type(e).__name__}: {e!s:.100}", "FAIL"
    _rows.append((name, status))
    print(f"  [{status:4}] {name:14} {detail}")
    if status == "FAIL" and lose:
        print(f"         lose: {lose}")


def _one_eog():
    return next(client.tasks(*EOG_PROBE, limit=1))


def _auth():
    """Prove the key works before anything expensive depends on it.

    A rejected key fails every row below this one, all with errors that look like
    the deployment is broken rather than like a credential problem -- so name it here.
    """
    req = urllib.request.Request(f"{SERVICE_URL}/v1/health",
                                 headers={"ngrok-skip-browser-warning": "true"})
    try:                                          # no key at all: is the gate even on?
        urllib.request.urlopen(req, timeout=30)
        gated = False
    except urllib.error.HTTPError as e:
        if e.code not in (401, 403, 503):
            raise
        gated = True
    client.benchmarks()                           # now WITH the key -- raises if rejected
    return ("key accepted" if gated else
            "key accepted (deployment is currently open -- no key required)")


def _service():
    h = client.health()
    if not h.get("ok"):
        raise RuntimeError(f"health.ok={h.get('ok')!r}")
    return f"ok, {h.get('active_sessions')} live session(s), ttl {h.get('ttl_sec')}s"


def _sdk():
    import simple_agentic_evals as m
    req = urllib.request.Request(f"{SERVICE_URL}/sdk",
                                 headers={"Authorization": f"Bearer {os.environ['EVAL_SERVICE_API_KEY']}",
                                          "ngrok-skip-browser-warning": "true"})
    served = re.search(r"-(\d+\.\d+\.\d+)-",
                       json.loads(sdk_read(req))["filename"]).group(1)
    mine = str(getattr(m, "__version__", "?"))
    if mine != served:                           # a stale client drifts from the API silently
        raise RuntimeError(f"client {mine} != served {served} -- re-run Setup")
    return f"{mine}, matches the served wheel"


def _catalog():
    kinds = {b["kind"] for ds in client.benchmarks()["datasets"]
             for b in ds["benchmarks"] if b.get("kind") in ("eog", "ale")}
    if {"eog", "ale"} - kinds:
        raise RuntimeError(f"catalog is missing {sorted({'eog', 'ale'} - kinds)}")
    return "eog + ale both advertised"


def _advertised(dataset, benchmark, version, split, domain):
    """What the catalog claims a stage holds, for cross-checking against task_ids."""
    for ds in client.benchmarks()["datasets"]:
        if ds["dataset"] != dataset:
            continue
        for b in ds["benchmarks"]:
            if b["benchmark"] != benchmark:
                continue
            for d in b["domains"]:
                if d["domain"] != domain:
                    continue
                for v in d["versions"]:
                    if v["version"] == version:
                        return v[f"n_{split}"]
    return None


def _eog_tasks():
    ids = client.task_ids(*EOG_PROBE)
    if not ids:
        raise RuntimeError("reference slice is empty")
    n = _advertised(*EOG_PROBE)                  # catalog and listing must agree
    if n != len(ids):
        raise RuntimeError(f"catalog advertises {n} tasks, task_ids returns {len(ids)}")
    return f"{len(ids)} tasks in {'/'.join(map(str, EOG_PROBE[1:]))}, matches the catalog"


def _ale_tasks():
    """ALE serves only what a published number can count -- verify the arithmetic."""
    m = client.health().get("ale_tasks") or {}
    if not m.get("ok"):
        raise RuntimeError(f"ALE task set unavailable: {m.get('reason')}")
    if not m.get("manifest_agrees"):
        raise RuntimeError(f"manifest drifted from ALE's task lists: {m.get('disagreements')}")
    if not m.get("filtered"):
        raise RuntimeError("this deployment sets EVAL_SERVICE_ALE_SERVE_ALL, so the catalog "
                           "includes tasks no published ALE number counts")
    served = set()
    for ds in client.benchmarks()["datasets"]:
        for b in ds["benchmarks"]:
            if b.get("kind") != "ale":
                continue
            for d in b["domains"]:
                for v in [x["version"] for x in d["versions"]]:
                    for sp in ("train", "test"):
                        served |= set(client.task_ids(ds["dataset"], "ale", v, sp,
                                                      d["domain"] or None))
    if len(served) != m["n_runnable"]:
        raise RuntimeError(f"catalog lists {len(served)} ale tasks, "
                           f"the manifest says {m['n_runnable']} are runnable")
    if m.get("example_filtered") in served:      # prove the filter is actually live
        raise RuntimeError(f"withheld task {m['example_filtered']} is still listed")
    return (f"{len(served)}/{m['n_suite']} runnable = {m['n_docker_support']} docker + "
            f"{m['n_privileged']} privileged; {m['n_excluded']} excluded + "
            f"{m['n_not_linux']} non-Linux withheld")


def _eog_env():
    task = _one_eog()
    with task:                                   # fresh DB, freed on exit
        mcp = task.mcp_session(task.mcp_servers[0])
        try:
            tools = mcp.list_tools()
        finally:
            mcp.close()
        if not tools:
            raise RuntimeError("gym returned 0 MCP tools")
        g = task.grade(keep_alive=True)          # no agent acted -> this is the baseline
    if not g.n_total:
        raise RuntimeError("grader ran 0 verifiers")
    return f"{len(tools)} tools, {g.n_total} verifiers, 1-task baseline {g.pass_rate:.2f}"


def _resources():
    tid = client.task_ids(*EOG_PROBE)[0]
    c = {m: client.resources(*EOG_PROBE[:3], task_id=tid, split=EOG_PROBE[3],
                             domain=EOG_PROBE[4], mode=m)["count"]
         for m in ("none", "oracle", "accumulative")}
    if c["accumulative"] < c["oracle"]:
        raise RuntimeError(f"accumulative {c['accumulative']} < oracle {c['oracle']}")
    return "  ".join(f"{k}={v}" for k, v in c.items())


def _harness_row(fn, **kw):
    task = _one_eog()
    with task:
        run = fn(task, api_key=os.environ["OPENAI_API_KEY"], **kw)
    stopped = getattr(run, "stopped", "?")
    if stopped == "error":                       # ran, but the turn failed on the server
        raise RuntimeError("harness returned stopped='error' -- see the service log")
    return f"ran on the service (stopped={stopped})"


def _ale():
    ad = client.health().get("ale_docker") or {}
    task = client.task(*ALE_PROBE, domain=None)  # ALE is flat -> domain=None
    with task:
        files = task.inputs()
        if not files:
            raise RuntimeError("no input files staged")
        task.fetch_input(files[0]["path"])
        # A stub artifact is enough: we're proving evaluate() executes, not scoring well.
        task.submit_text(task.output_path or "output/agent_output.json", "{}")
        g = task.grade(keep_alive=True)
    return (f"{len(files)} inputs, evaluate() ran (stub -> {g.pass_rate}), "
            f"sandbox={ad.get('enabled')} dind={ad.get('dind_available')}")


print("deployment:", SERVICE_URL)
_check("auth",          _auth,      "everything -- get a key from MyAuthtoken")
_check("reachable",     _service,   "everything")
_check("sdk",           _sdk,       "silent API drift -- re-run Setup")
_check("catalog",       _catalog,   "a benchmark isn't deployed here")
_check("eog tasks",     _eog_tasks, "this EOG slice")
_check("ale tasks",     _ale_tasks, "a trustworthy ALE denominator (5)")
_check("eog env+grade", _eog_env,   "all of EOG (2)")
_check("resources",     _resources, "the evolving modes (6)")
_check("react",         lambda: _harness_row(react_agent, model=LLM_MODEL, max_steps=1),
       "the ReAct harness -- your own loop (2) still works", deep=True)
_check("codex",         lambda: _harness_row(acp_codex_agent, max_episodes=1),
       "the Codex harness -- your own loop (2) still works", deep=True)
_check("ale",           _ale,       "all of ALE (5)", deep=True)

skip = [n for n, s in _rows if s == "SKIP"]
red = [n for n, s in _rows if s == "FAIL"]
ok = sum(1 for _, s in _rows if s == "OK")
line = f"\n{ok}/{len(_rows) - len(skip)} green"
line += f"   RED: {', '.join(red)}" if red else "   every feature on this deployment works"
print(line + (f"   (skipped: {', '.join(skip)})" if skip else ""))

## 1. Wrap your method

`run_benchmark` takes **anything callable that accepts a task**. There is no base class to
subclass and nothing to register — if your method already exists, you are writing an adapter,
not a port.

```python
def my_agent(task):
    ...            # act on the task
```

Three rules, and everything else is yours:

1. **Return nothing the grader needs.** It never reads your return value; it reads the
   environment you acted on. What you leave in the gym database (EOG) or in the sandbox
   filesystem (ALE) *is* your answer.
2. **Close what you open.** `mcp.close()` before returning, from a `finally:` — a crash
   mid-task should still release the session.
3. **Report your tokens** if you want the cost columns filled. Your model calls never touch the
   service, so we cannot count them. `return {"total_tokens": n}` and they appear in the row;
   omit it and the row says `not measured` rather than a misleading `0`.

You never write `with task:` — `run_benchmark` provisions a fresh environment before each call
and tears it down after.

### The two action surfaces

A leaderboard row covers both benchmarks, and they hand your agent different affordances. This
is the whole of the API you need:

| | **EOG** — act through tools | **ALE** — produce an artifact |
|---|---|---|
| what you get | `task.mcp_servers`, `task.mcp_session(s)` | `task.inputs()`, `task.fetch_input(path)` |
| a whole directory | — | `task.fetch_inputs_to(dir)`, `task.submit_dir(dir)` |
| how you act | call gym tools over MCP | write files: `task.submit_text(path, text)` |
| what is graded | final database state, hidden SQL verifiers | the task's own `evaluate()` over your files |
| where to write | — | `task.output_path` |

Both give you `task.system_prompt` and `task.user_prompt`. One function can serve both by
branching on `task.mcp_servers`, which is what the example below does.

Both give you `task.system_prompt` and `task.user_prompt`. One function can serve both by
branching on `task.mcp_servers`, which is what the example below does.

In [ ]:
def solve_eog(task, max_steps=6):
    """EOG: act on the gym through MCP tools. Replace the loop with your method."""
    used = 0
    mcp = task.mcp_session(task.mcp_servers[0])
    try:
        tools = to_openai_tools(mcp.list_tools())          # MCP schema -> OpenAI schema
        msgs = [{"role": "system", "content": task.system_prompt or ""},
                {"role": "user",   "content": task.user_prompt or ""}]
        for _ in range(max_steps):
            r = oai.chat.completions.create(model=LLM_MODEL, messages=msgs, tools=tools)
            used += r.usage.total_tokens
            m = r.choices[0].message
            msgs.append(m.model_dump(exclude_none=True))
            if not m.tool_calls:                            # no call -> the agent is done
                break
            for tc in m.tool_calls:
                out = mcp.call_tool(tc.function.name,
                                    json.loads(tc.function.arguments or "{}"))
                msgs.append({"role": "tool", "tool_call_id": tc.id,
                             "content": json.dumps(out)[:4000]})
    finally:
        mcp.close()                                         # rule 2 -- even if we raised
    return {"total_tokens": used}


def solve_ale(task):
    """ALE: read the staged inputs, write the deliverable. One shot, no tools."""
    files = task.inputs()
    seen = [f["path"] for f in files][:20]
    head = task.fetch_input(files[0]["path"])[:4000] if files else ""
    r = oai.chat.completions.create(model=LLM_MODEL, messages=[
        {"role": "system", "content": task.system_prompt or ""},
        {"role": "user", "content": f"{task.user_prompt}\n\nFiles: {seen}\n\n{head}"}])
    task.submit_text(task.output_path or "output/agent_output.json",
                     r.choices[0].message.content or "")
    return {"total_tokens": r.usage.total_tokens}


def my_agent(task, max_steps=6):
    """One entry point, both benchmarks. This is the function you swap out."""
    if task.mcp_servers:                                    # EOG tasks expose gym servers
        return solve_eog(task, max_steps=max_steps)
    return solve_ale(task)                                  # ALE tasks stage files instead

## 2. How your policy connects

Whatever you already have — a prompt optimizer, a memory, a multi-agent scaffold, a container,
or a CLI — can connect either through the downloadable skill or through the Python wrapper.

### If you just want a quick dry run

A shell-capable agent can install the skill itself. Set the eval key before downloading it:

```bash
mkdir -p ~/.codex/skills/evolve-eval
: "${EVAL_SERVICE_API_KEY:?Required — sign in at https://mas-orchestra.salesforceresearch.ai/mas_r1/demo/ and open MyAuthtoken}" &&
curl -fsSL -H "Authorization: Bearer $EVAL_SERVICE_API_KEY" \
  "https://educator-marrow-cultural.ngrok-free.dev/resources/evolve-eval/SKILL.md" \
  -o ~/.codex/skills/evolve-eval/SKILL.md
```

Then ask the current coding agent to use the `evolve-eval` skill to evaluate itself. The agent
handles `start`, works on the task through its tools or files, calls `grade`, and returns the
result. The same `EVAL_SERVICE_API_KEY` authenticates the download and evaluation calls, so the user does not need to run
each command manually. This interactive path is a quick rehearsal only: its result is always
marked `partial`, not a scalable or publishable leaderboard evaluation.

### The wrapper

```python
def my_agent(task) -> dict | None:
```

Called once per task. We provision the environment before the call and grade after it; the
return value is optional telemetry, never the answer.

Your policy runs in **your** process and calls the service, so its language, framework and
runtime stay yours. Anything reachable from inside a Python function can be scored.

### EOG tool-use interface — MCP

`task.mcp_url(server)` with an `Authorization: Bearer` header is a standard **MCP endpoint over
streamable HTTP**, so any conformant MCP client connects to it — the official MCP SDK included:

```python
server  = task.mcp_servers[0]
url     = task.mcp_url(server)
headers = {"Authorization": f"Bearer {EVAL_SERVICE_API_KEY}", **server.headers}
```

Couple at whatever level your policy already speaks:

| your policy speaks | how it connects | glue |
|---|---|---|
| **MCP** natively | point its own client at the url and headers above | none |
| **OpenAI** function calling | `to_openai_tools(mcp.list_tools())`, as in §1 | one line |
| **anything else** | the two calls in the next cell | a few lines |

The third row is the contract the other two are built on:

> **A list of JSON-Schema tool declarations, plus a `call(name, args)` function.**

Every agent framework consumes that pair — OpenAI tools, Anthropic tools, LangChain tools and
MCP itself are all that same shape — which is why a policy we have never seen still plugs in.

The grader then reads the environment your policy changed.


In [ ]:
# EOG tool-use interface, in the plainest form: list tools, call one, close the session.
# These two calls are the entire seam -- everything else is a convenience on top.
probe = next(client.tasks("evovling_tools", "eog", 1, "test", "hr", limit=1))

with probe:                                    # provisions a fresh environment for the task
    mcp = probe.mcp_session(probe.mcp_servers[0])
    try:
        tools = mcp.list_tools()               # 1. WHAT YOUR POLICY CAN DO.
        print(f"{len(tools)} tools on this gym. The first declaration looks like:\n")
        print(json.dumps(tools[0], indent=2)[:360], "...\n")

        result = mcp.call_tool(tools[0]["name"], {})   # 2. HOW YOUR POLICY ACTS.
        print("call_tool returned:", str(result)[:150])
    finally:
        mcp.close()                            # rule 2, always

# `tools` is plain JSON Schema and `call_tool` is a plain function, so handing this pair
# to a foreign agent loop is all the integration a new framework ever needs.

### ALE artifact-delivery interface — input/output files

`fetch_inputs_to(dir)` stages the task's files on your disk, and `submit_dir(dir)` ships back
whatever your policy wrote. The contract is a filesystem, so a container, a CLI, a remote job or
another language all sit behind it equally well.

Submitted paths are normalized under the task's `output/`, so pointing `submit_dir` at the
sandbox root or at the output directory itself both land correctly, and non-UTF-8 files are
base64-encoded automatically so binaries survive.


In [ ]:
import subprocess, tempfile
from pathlib import Path


def containerized_agent(task):
    """ALE file interface: an existing container, unchanged, scored on an ALE task.

    Your container never talks to the service. It reads one directory and writes
    another, exactly as it does on your laptop -- we fill the first and collect
    the second.
    """
    with tempfile.TemporaryDirectory() as tmp:
        work   = Path(tmp)          # a scratch directory on YOUR machine
        indir  = work / "input"     # we fill this one
        outdir = work / "output"    # your system fills this one
        outdir.mkdir()

        task.fetch_inputs_to(indir)                    # 1. service -> your disk

        subprocess.run([                               # 2. your system runs, however it likes
            "docker", "run", "--rm",
            "-v", f"{work}:/work",                     #    `work` shows up as /work inside
            "my-system:latest",
            "--input", "/work/input",                  #    the same two dirs, container-side
            "--output", "/work/output",
        ], check=True, timeout=1800)

        task.submit_dir(outdir)                        # 3. your disk -> service, then grading
    return None  # omit telemetry when this wrapper cannot measure tokens or steps


# Nothing to do with Docker, really. For a plain CLI, steps 1 and 3 are identical and
# step 2 becomes:
#     subprocess.run(["my-system", "--in", indir, "--out", outdir], check=True)

## 3. Smoke test it

Two tasks on each benchmark, a few cents, a couple of minutes. The point is only to prove your
adapter runs and the grader sees what you left behind — **these numbers are not comparable to
anything**, and the report says so with a `SUBSAMPLED` warning in the scope line so a debug run
can never be mistaken for a result.

If `solved` is 0 but `ACC` is not, your agent is getting partial credit — normal for two tasks.
If `latency` is a fraction of a second and `ACC` sits exactly at the floor, your agent probably
is not acting at all; §4 shows you what the floor looks like.

In [ ]:
for bench in ("eog", "ale"):
    rep = run_benchmark(my_agent, mode="deployment_eval", client=client,
                        benchmark=bench, limit=2, progress=True)
    print("\n" + str(rep) + "\n")
    print("per-task verifier results:")
    for result in rep.task_results:
        print(result["task_id"], result.get("per_verifier", []))

## 4. Sanity: what does doing nothing score?

Some verifiers pass on the seeded state alone, so the real zero of this benchmark is not `0.0`.
Knowing the floor is what makes your own number readable — and it is the score an agent returns
when it silently fails to act, which is the most common way a run lies to you.

In [ ]:
def floor_agent(task):
    """Acts on nothing. Whatever this scores, the environment was giving away."""
    return {"total_tokens": 0}


floor = run_benchmark(floor_agent, mode="deployment_eval", client=client,
                      benchmark="eog", limit=2, progress=False)
mine  = run_benchmark(my_agent,    mode="deployment_eval", client=client,
                      benchmark="eog", limit=2, progress=False)
print(f"  do-nothing floor   ACC {floor.accuracy:.3f}   solved {floor.success_rate:.3f}")
print(f"  my_agent           ACC {mine.accuracy:.3f}   solved {mine.success_rate:.3f}")
print("\n^ if these two match, your agent is not acting -- check that it reaches mcp.call_tool")

## 5. Produce the leaderboard row

This is the whole point of the notebook. SDK `run_leaderboard` runs both benchmarks, repeats each
run so the row carries a spread, and returns **exactly the record the leaderboard stores** — the
same eight columns plus the overall rate, in the same field names.

Three details that make the numbers comparable rather than merely plausible:

- **Hours are summed agent duration, not wall clock.** Provisioning and grading are ours, not
  yours, so they do not count against you. That is `report.agent_s`, not `report.latency_s`.
- **± is the population standard deviation over repeated runs**, not a standard error and not a
  confidence interval. The published rows use 3.
- **Overall pools tasks, not percentages.** It is total solved over total attempted across both
  benchmarks, so EOG's 454 tasks outweigh ALE's 63 — a macro-average of the two rates would be
  a different, larger number.

In [ ]:
# Rehearsal: exactly 2 tasks per benchmark, 2 repeats. Prove the pipeline, not the method.
row = run_leaderboard(
    my_agent, f"my_agent ({LLM_MODEL})", client=client, seeds=2, limit=2,
    benchmark_kwargs={"eog": {"domain": "hr"}, "ale": {"domain": None}},
)
print(json.dumps(row, indent=2))

## 6. The publishable run

Drop `limit`, set `seeds=3`, and the row becomes comparable. Before you start it, size it:

| | EOG | ALE |
|---|---|---|
| tasks per run (`evovling_tools`) | 454 | 63 |
| × 3 repeats | 1,362 | 189 |
| published rows spend | 21–98 agent-hours | 6–17 agent-hours |
| published rows spend | 27–179 M tokens | 20–184 M tokens |

That is your budget, not ours — the service provisions and grades for free, but every token is
billed to the key you supplied. **Rehearse with `limit` until the row shape looks right**, then
run the real thing once.

```python
row = run_leaderboard(my_agent, "My Method", client=client, cat="memory", seeds=3,
                      confirm_full_cost=True)  # explicit acknowledgement: hours, real money
print(json.dumps(row, indent=2))
```

Two switches worth knowing before you commit:

- `dataset=` picks the evolving axis: `"evovling_tools"` (the main table), `"evovling_skills"`,
  or `"evovling_agents"`. Each is a separate leaderboard.
- `mode=` picks the question. `"deployment_eval"` is a fixed method under a growing harness —
  the right default and the one the rows above use. If your method *learns* between stages
  (memory, prompt evolution, self-modifying code), it belongs in
  `"self_evolving_adapt_eval"`, which needs an `adapt` step — that is Level 2.
- `cat=` is the leaderboard's family grouping: `"deploy"`, `"memory"`, `"prompt"`, `"code"`.
  Set `mas=True` if your method is a multi-agent system.

When the run finishes, the printed JSON is the submission. Send it with the config that produced
it — model, dataset, mode, and the `seeds` count.

## Cheat sheet

```python
report = run_benchmark(my_agent, mode="deployment_eval", client=client, benchmark="eog")
```

| argument | what it does |
|---|---|
| `agent` | your `f(task)`, or `"react"` / `"codex"` for a provided harness (Level 3) |
| `mode` | `"deployment_eval"` fixed method · `"self_evolving_adapt_eval"` it adapts (Level 2) · `"task_specific"` oracle reference |
| `dataset` | `"evovling_tools"` (default) · `"evovling_skills"` · `"evovling_agents"` |
| `benchmark` | `"eog"` (default) tool calling · `"ale"` artifacts |
| `domain` | one domain or a list — omit for all of them |
| `limit` | tasks per cohort; **rehearsal only, drop it for a real row** |
| `agent_kwargs` | forwarded to your function or the harness |
| `matrix=True` | fill the whole matrix → BWT / FWT (Level 2) |
| `on_error` | `"score_zero"` (default) or `"raise"` while debugging |

| on the report | the leaderboard column it feeds |
|---|---|
| `.success_rate` | **Pass %** — every verifier passed |
| `.accuracy` | **Score** — mean verifier pass rate |
| `.agent_s` | **⏱ h** — summed agent duration (`.latency_s` is wall clock, not this) |
| `.total_tokens` | **Tok M** |
| `.n_tasks` `.n_success` | what **Overall** pools |
| `.n_errors` `.complete` | failures; whether `limit` subsampled it |
| `.last_row` | per-domain `CohortResult`s |
| `.last_row[i].per_task` | every task's `score`, `solved`, `latency_s`, `total_tokens`, `per_verifier` |
| `.task_results` | all final-row task results flattened across domains |
| `.verifier_results` | one flattened row per task × verifier, with `task_id`, name, score, and pass status |

| on an ALE task | |
|---|---|
| `.fetch_inputs_to(dir)` | stage every input file on your disk, relative paths kept |
| `.submit_dir(dir)` | submit everything under `dir`; binaries base64'd automatically |
| `.inputs()` `.fetch_input(p)` `.submit_text(p, s)` | the per-file versions |
| `.output_path` | the deliverable path the prompt asks for |

**Next:** [Level 2 · Continual learning](https://colab.research.google.com/drive/1vrcGelN9GmwiCK25c6qZNG3Z0sHWxE5o)
— the full performance matrix, retention and adaptation, BWT and FWT, and methods that learn
between stages.